# Data 304 — NLP Wrangling Demo

This notebook mirrors the lecture demos:
- Tokenization
- POS and dependencies
- Dependency visualization (displaCy)
- Named Entity Recognition (NER)
- Chunking (noun phrases)
- Converting annotations to a DataFrame
- Quantitative entity summary
- Custom rules with `EntityRuler`
- Bonus: simple verb-phrase patterns with `Matcher`

## Environment setup
Run this once if needed:
```bash
pip install spacy pandas matplotlib
python -m spacy download en_core_web_sm
```


In [ ]:
# Imports and model load
import spacy
from spacy import displacy
from spacy.matcher import Matcher
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    raise OSError(
        "Model 'en_core_web_sm' is not installed. Run: python -m spacy download en_core_web_sm"
    )

TEXT = "UT Knoxville launched a new Data Science program in Fall 2025. Python processes data efficiently. Efficient data processing methods improve performance."
doc = nlp(TEXT)
TEXT

## Tokenization (Slide 4)
Convert text into tokens.

In [ ]:
[t.text for t in doc]

## POS and Dependencies (Slides 5–6)
`.pos_` gives universal POS; `.dep_` gives dependency labels.

In [ ]:
for token in doc:
    print(token.text, token.pos_, token.dep_)

### Common dependency labels in our examples
- **nsubj**: nominal subject
- **ROOT**: main governing word of the sentence (usually the main verb)
- **aux**: auxiliary verb
- **acomp**: adjectival complement
- **dobj** / **obj**: direct object (label differs across versions)
- **advmod**: adverbial modifier

## Visualizing Dependencies
Renders inline in Jupyter. Use `displacy.serve(doc, style="dep")` for a local web app.

In [ ]:
# from spacy import displacy
# displacy.render(doc, style="dep")

In [ ]:
from spacy import displacy
html = displacy.render(doc, style="dep", jupyter=False)  # avoids IPython import
open("deps.html","w",encoding="utf-8").write(html)

## Named Entity Recognition (Slides 8–10)
Inspect entities, labels, and spans.

In [ ]:
[(ent.text, ent.label_) for ent in doc.ents]

In [ ]:
for ent in doc.ents:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

In [ ]:
# View entity label set for the current model
nlp.get_pipe("ner").labels

## Chunking and Phrases (Slide 11)
spaCy exposes *noun chunks* directly.

In [ ]:
list(chunk.text for chunk in doc.noun_chunks)

### Bonus: Simple verb phrase patterns with `Matcher`
This is a naive pattern: VERB optionally followed by one or more NOUNs.

In [ ]:
matcher = Matcher(nlp.vocab)
pattern = [{"POS": "VERB"}, {"POS": "NOUN", "OP": "*"}]
matcher.add("VERB_PHRASE", [pattern])
matches = matcher(doc)
[(doc[start:end].text, start, end) for _, start, end in matches]

## From Tokens to Features (Slide 12)
Build a table of linguistic annotations.

In [ ]:
rows = [(t.text, t.lemma_, t.pos_, t.dep_, t.ent_type_) for t in doc]
df = pd.DataFrame(rows, columns=["Token", "Lemma", "POS", "Dep", "Entity"]) 
df

## Quantitative Entity Summary (Slide 15)
Count and visualize entity distribution across the text.

In [ ]:
counts = Counter(ent.label_ for ent in doc.ents)
counts

In [ ]:
if counts:
    labels, values = zip(*counts.items())
    plt.figure()
    plt.bar(labels, values)
    plt.title("Entity label counts")
    plt.xlabel("Label")
    plt.ylabel("Count")
    plt.show()
else:
    print("No entities in text.")

## Custom Entity Rules (Slide 19)
Use `EntityRuler` to add domain-specific entities (e.g., skills).

In [ ]:
from spacy.pipeline import EntityRuler
ruler = nlp.add_pipe("entity_ruler", before="ner")
patterns = [
    {"label": "SKILL", "pattern": "Python"},
    {"label": "SKILL", "pattern": "Pandas"}
]
ruler.add_patterns(patterns)

doc2 = nlp("Python and Pandas are popular in data science.")
[(ent.text, ent.label_) for ent in doc2.ents]

---
## Appendix: Utility helpers
- Re-run `doc = nlp(TEXT)` after you change `TEXT`.
- Use `displacy.serve(doc, style="dep")` for a standalone web view.
- Save tables: `df.to_csv("annotations.csv", index=False)`
Generated: 2025-11-06T14:19:55.034602Z
